In [3]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# 定义数据文件路径
ratings_file = r"F:\zhoumian\book\ml-1m\ratings.dat"

# 读取ratings.dat文件
def load_ratings(file_path):
    with open(file_path, "r") as file:
        data = file.readlines()
    transactions = {}
    for line in data:
        user_id, movie_id, rating, timestamp = line.strip().split("::")
        user_id, movie_id = int(user_id), int(movie_id)
        if user_id not in transactions:
            transactions[user_id] = []
        transactions[user_id].append(movie_id)
    return list(transactions.values())

# 加载用户观看电影的事务数据
transactions = load_ratings(ratings_file)

# 使用TransactionEncoder将事务数据转为布尔值编码
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

# 使用FP-Growth算法挖掘频繁项集
min_support = 0.4  # 设置最小支持度阈值（根据需要调整）
frequent_itemsets = fpgrowth(df, min_support=min_support, use_colnames=True)

# 为频繁项集添加支持度列
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(lambda x: len(x))

# 打印频繁项集
print("频繁项集（支持度大于0.01）：")
print(frequent_itemsets)

# 保存结果到文件
output_file = "frequent_itemsets.csv"
frequent_itemsets.to_csv(output_file, index=False)
print(f"频繁项集已保存到 {output_file}")


频繁项集（支持度大于0.01）：
     support itemsets  length
0   0.495199    (260)       1
1   0.439238   (2028)       1
2   0.427649   (1270)       1
3   0.416060    (608)       1
4   0.407119   (2762)       1
5   0.567550   (2858)       1
6   0.495033   (1196)       1
7   0.477318   (1210)       1
8   0.442384    (480)       1
9   0.438576    (589)       1
10  0.428808   (2571)       1
11  0.426821    (593)       1
12  0.416225   (1198)       1
13  0.404470    (110)       1
14  0.420199   (1580)       1
频繁项集已保存到 frequent_itemsets.csv


In [7]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
import chardet

# 定义文件路径
users_file = r"F:\zhoumian\book\ml-1m\users.dat"
ratings_file = r"F:\zhoumian\book\ml-1m\ratings.dat"
movies_file = r"F:\zhoumian\book\ml-1m\movies.dat"

# 检测文件编码
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

# 加载文件的通用函数
def load_file(file_path, sep, names, encoding=None):
    if encoding is None:
        encoding = detect_encoding(file_path)
    try:
        data = pd.read_csv(
            file_path,
            sep=sep,
            header=None,
            names=names,
            engine="python",
            encoding=encoding
        )
    except UnicodeDecodeError:
        # 如果编码错误，尝试使用常见的 ISO-8859-1 编码
        print(f"UnicodeDecodeError: 尝试使用 ISO-8859-1 编码重新加载 {file_path}")
        data = pd.read_csv(
            file_path,
            sep=sep,
            header=None,
            names=names,
            engine="python",
            encoding="ISO-8859-1"
        )
    return data

# 加载数据
users = load_file(users_file, sep="::", names=["UserID", "Gender", "Age", "Occupation", "Zip-code"])
ratings = load_file(ratings_file, sep="::", names=["UserID", "MovieID", "Rating", "Timestamp"])
movies = load_file(movies_file, sep="::", names=["MovieID", "Title", "Genres"])

# 按年龄或职业分组
age_group = 18  # 示例：18-24 岁年龄段
occupation_group = 12  # 示例：职业为程序员（Programmer）
filtered_users = users[(users["Age"] == age_group) & (users["Occupation"] == occupation_group)]

# 筛选分组用户的观影记录
filtered_ratings = ratings[ratings["UserID"].isin(filtered_users["UserID"])]

# 构建观影事务数据（每位用户看过的电影集合）
transactions = filtered_ratings.groupby("UserID")["MovieID"].apply(list).tolist()

# 使用TransactionEncoder转换事务数据
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

# 使用FP-Growth挖掘频繁项集
min_support = 0.5  # 设置最小支持度（根据需要调整）
frequent_itemsets = fpgrowth(df, min_support=min_support, use_colnames=True)

# 将MovieID映射为电影标题
movie_id_to_title = dict(zip(movies["MovieID"], movies["Title"]))
frequent_itemsets["itemsets"] = frequent_itemsets["itemsets"].apply(
    lambda x: {movie_id_to_title[int(i)] for i in x}
)

# 打印结果
print("频繁项集：")
print(frequent_itemsets)

# 保存结果到文件
output_file = "frequent_itemsets_by_group.csv"
frequent_itemsets.to_csv(output_file, index=False)
print(f"频繁项集已保存到 {output_file}")


频繁项集：
     support                                           itemsets
0   0.677966                               {Matrix, The (1999)}
1   0.661017  {Star Wars: Episode V - The Empire Strikes Bac...
2   0.644068                           {American Beauty (1999)}
3   0.644068                             {Jurassic Park (1993)}
4   0.610169  {Star Wars: Episode VI - Return of the Jedi (1...
5   0.593220                {Terminator 2: Judgment Day (1991)}
6   0.559322  {Star Wars: Episode I - The Phantom Menace (19...
7   0.508475                                 {Gladiator (2000)}
8   0.576271                       {Princess Bride, The (1987)}
9   0.610169                          {Sixth Sense, The (1999)}
10  0.525424                                {Braveheart (1995)}
11  0.593220        {Star Wars: Episode IV - A New Hope (1977)}
12  0.525424                                {Fight Club (1999)}
13  0.508475                      {Being John Malkovich (1999)}
14  0.542373  {Matrix, The (1999),

In [8]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
import chardet

# 定义文件路径
users_file = r"F:\zhoumian\book\ml-1m\users.dat"
ratings_file = r"F:\zhoumian\book\ml-1m\ratings.dat"
movies_file = r"F:\zhoumian\book\ml-1m\movies.dat"

# 检测文件编码
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

# 加载文件的通用函数
def load_file(file_path, sep, names, encoding=None):
    if encoding is None:
        encoding = detect_encoding(file_path)
    try:
        data = pd.read_csv(
            file_path,
            sep=sep,
            header=None,
            names=names,
            engine="python",
            encoding=encoding
        )
    except UnicodeDecodeError:
        print(f"UnicodeDecodeError: 尝试使用 ISO-8859-1 编码重新加载 {file_path}")
        data = pd.read_csv(
            file_path,
            sep=sep,
            header=None,
            names=names,
            engine="python",
            encoding="ISO-8859-1"
        )
    return data

# 加载数据
users = load_file(users_file, sep="::", names=["UserID", "Gender", "Age", "Occupation", "Zip-code"])
ratings = load_file(ratings_file, sep="::", names=["UserID", "MovieID", "Rating", "Timestamp"])
movies = load_file(movies_file, sep="::", names=["MovieID", "Title", "Genres"])

# 按年龄或职业分组
age_group = 18  # 示例：18-24 岁年龄段
occupation_group = 12  # 示例：职业为程序员（Programmer）
filtered_users = users[(users["Age"] == age_group) & (users["Occupation"] == occupation_group)]

# 筛选分组用户的观影记录
filtered_ratings = ratings[ratings["UserID"].isin(filtered_users["UserID"])]

# 构建观影事务数据（每位用户看过的电影集合）
transactions = filtered_ratings.groupby("UserID")["MovieID"].apply(list).tolist()

# 使用TransactionEncoder转换事务数据
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

# 使用FP-Growth挖掘频繁项集
min_support = 0.3  # 设置最小支持度
frequent_itemsets = fpgrowth(df, min_support=min_support, use_colnames=True)

# 将MovieID映射为电影标题
movie_id_to_title = dict(zip(movies["MovieID"], movies["Title"]))
frequent_itemsets["itemsets"] = frequent_itemsets["itemsets"].apply(
    lambda x: {movie_id_to_title[int(i)] for i in x}
)

# 打印频繁项集
print("频繁项集：")
print(frequent_itemsets)

# 生成关联规则
min_confidence = 0.6  # 设置最小置信度阈值
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)

# 打印关联规则
print("强关联规则：")
print(rules)

# 保存结果到文件
output_itemsets_file = "frequent_itemsets_by_group.csv"
output_rules_file = "association_rules_by_group.csv"
frequent_itemsets.to_csv(output_itemsets_file, index=False)
rules.to_csv(output_rules_file, index=False)

print(f"频繁项集已保存到 {output_itemsets_file}")
print(f"强关联规则已保存到 {output_rules_file}")


频繁项集：
       support                                           itemsets
0     0.677966                               {Matrix, The (1999)}
1     0.661017  {Star Wars: Episode V - The Empire Strikes Bac...
2     0.644068                           {American Beauty (1999)}
3     0.644068                             {Jurassic Park (1993)}
4     0.610169  {Star Wars: Episode VI - Return of the Jedi (1...
...        ...                                                ...
1705  0.305085              {Sneakers (1992), Matrix, The (1999)}
1706  0.305085  {Jurassic Park (1993), Sneakers (1992), Matrix...
1707  0.305085  {Pulp Fiction (1994), Professional, The (a.k.a...
1708  0.305085  {Blade Runner (1982), Star Wars: Episode V - T...
1709  0.305085          {Matrix, The (1999), Blade Runner (1982)}

[1710 rows x 2 columns]
强关联规则：
                                             antecedents  \
0                                   (Matrix, The (1999))   
1      (Star Wars: Episode V - The Empire Strikes 

In [10]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Replace the file paths with the correct paths to your data files
users_file = "F:\\zhoumian\\book\\ml-1m\\users.dat"
ratings_file = "F:\\zhoumian\\book\\ml-1m\\ratings.dat"

# Load users data
users = pd.read_csv(
    users_file,
    sep="::",
    header=None,
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip-code"],
)

# Load ratings data
ratings = pd.read_csv(
    ratings_file,
    sep="::",
    header=None,
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

# Merge users and ratings data on UserID
data = pd.merge(users, ratings, on="UserID")

# Choose whether to group by 'Age' or 'Occupation'
group_by_column = "Age"  # Change to "Occupation" if needed

# Get unique groups
groups = data[group_by_column].unique()

# Define minimum support and confidence thresholds
min_support = 0.3
min_confidence = 0.5

# Iterate over each group
for group in groups:
    print(f"Processing {group_by_column} group: {group}")
    
    # Filter data for the current group
    group_data = data[data[group_by_column] == group]
    
    # Create transactions: list of MovieIDs for each UserID
    transactions = group_data.groupby("UserID")["MovieID"].apply(list).values.tolist()
    
    # Apply TransactionEncoder to transform the transaction data
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    # Apply the FP-Growth algorithm to find frequent itemsets
    frequent_itemsets = fpgrowth(df_trans, min_support=min_support, use_colnames=True)
    
    if frequent_itemsets.empty:
        print(f"No frequent itemsets found for {group_by_column} group {group}")
        continue
    
    # Generate strong association rules from the frequent itemsets
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    
    if rules.empty:
        print(f"No association rules found for {group_by_column} group {group}")
        continue
    
    # Display the association rules
    print(f"Association rules for {group_by_column} group {group}:")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
    print("\n")


Processing Age group: 1
Association rules for Age group 1:
     antecedents   consequents   support  confidence      lift
0            (1)        (3114)  0.333333    0.660714  1.481602
1         (3114)           (1)  0.333333    0.747475  1.481602
2            (1)         (588)  0.310811    0.616071  1.609034
3          (588)           (1)  0.310811    0.811765  1.609034
4            (1)        (1580)  0.301802    0.598214  1.328036
5         (1580)           (1)  0.301802    0.670000  1.328036
6         (1210)         (260)  0.346847    0.770000  1.692475
7          (260)        (1210)  0.346847    0.762376  1.692475
8         (2571)        (1580)  0.315315    0.707071  1.569697
9         (1580)        (2571)  0.315315    0.700000  1.569697
10        (2571)         (260)  0.306306    0.686869  1.509751
11         (260)        (2571)  0.306306    0.673267  1.509751
12        (1210)        (2571)  0.301802    0.670000  1.502424
13        (2571)        (1210)  0.301802    0.676768  1.502

In [15]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 替换为您的数据文件路径
users_file = "F:\\zhoumian\\book\\ml-1m\\users.dat"
ratings_file = "F:\\zhoumian\\book\\ml-1m\\ratings.dat"
movies_file = "F:\\zhoumian\\book\\ml-1m\\movies.dat"

# 加载用户数据
users = pd.read_csv(
    users_file,
    sep="::",
    header=None,
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip-code"],
)

# 加载评分数据
ratings = pd.read_csv(
    ratings_file,
    sep="::",
    header=None,
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

# 加载电影数据
movies = pd.read_csv(
    movies_file,
    sep="::",
    header=None,
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding='latin-1'  # 处理特殊字符
)

# 创建MovieID到Title的映射字典
movie_dict = movies.set_index("MovieID")["Title"].to_dict()

# 合并用户和评分数据
data = pd.merge(users, ratings, on="UserID")

# 选择分组依据：'Age'或'Occupation'
group_by_column = "Age"  # 如有需要，可改为"Occupation"

# 获取唯一的分组值
groups = data[group_by_column].unique()

# 定义最小支持度和置信度阈值
min_support = 0.3
min_confidence = 0.5

# 定义函数，将关联规则中的MovieID替换为电影名称
def replace_movie_ids(rules):
    rules = rules.copy()
    rules['antecedents'] = rules['antecedents'].apply(lambda x: frozenset([movie_dict.get(i, f"MovieID {i}") for i in x]))
    rules['consequents'] = rules['consequents'].apply(lambda x: frozenset([movie_dict.get(i, f"MovieID {i}") for i in x]))
    return rules

# 遍历每个分组
for group in groups:
    print(f"正在处理 {group_by_column} 分组: {group}")
    
    # 筛选当前分组的数据
    group_data = data[data[group_by_column] == group]
    
    # 创建交易记录：每个用户的MovieID列表
    transactions = group_data.groupby("UserID")["MovieID"].apply(list).values.tolist()
    
    # 使用TransactionEncoder将交易数据转换为适合挖掘的格式
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    # 应用FP-Growth算法找到频繁项集
    frequent_itemsets = fpgrowth(df_trans, min_support=min_support, use_colnames=True)
    
    if frequent_itemsets.empty:
        print(f"{group_by_column} 分组 {group} 没有找到频繁项集")
        continue
    
    # 从频繁项集生成关联规则
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    
    if rules.empty:
        print(f"{group_by_column} 分组 {group} 没有找到关联规则")
        continue
    
    # 将MovieID替换为电影名称
    rules = replace_movie_ids(rules)
    
    # 显示关联规则
    print(f"{group_by_column} 分组 {group} 的关联规则:")
    print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
    print("\n")


正在处理 Age 分组: 1
Age 分组 1 的关联规则:
                                          antecedents  \
0                                  (Toy Story (1995))   
1                                (Toy Story 2 (1999))   
2                                  (Toy Story (1995))   
3                                    (Aladdin (1992))   
4                                  (Toy Story (1995))   
5                               (Men in Black (1997))   
6   (Star Wars: Episode VI - Return of the Jedi (1...   
7         (Star Wars: Episode IV - A New Hope (1977))   
8                                (Matrix, The (1999))   
9                               (Men in Black (1997))   
10                               (Matrix, The (1999))   
11        (Star Wars: Episode IV - A New Hope (1977))   
12  (Star Wars: Episode VI - Return of the Jedi (1...   
13                               (Matrix, The (1999))   
14  (Star Wars: Episode VI - Return of the Jedi (1...   
15  (Star Wars: Episode V - The Empire Strikes Bac...   


Age 分组 35 的关联规则:
                                          antecedents  \
0   (Star Wars: Episode V - The Empire Strikes Bac...   
1         (Star Wars: Episode IV - A New Hope (1977))   
2   (Star Wars: Episode V - The Empire Strikes Bac...   
3                         (Back to the Future (1985))   
4         (Star Wars: Episode IV - A New Hope (1977))   
5                         (Back to the Future (1985))   
6   (Star Wars: Episode V - The Empire Strikes Bac...   
7                 (Terminator 2: Judgment Day (1991))   
8         (Star Wars: Episode IV - A New Hope (1977))   
9                 (Terminator 2: Judgment Day (1991))   
10                             (Jurassic Park (1993))   
11                (Terminator 2: Judgment Day (1991))   
12                             (Jurassic Park (1993))   
13  (Star Wars: Episode V - The Empire Strikes Bac...   
14                             (Jurassic Park (1993))   
15        (Star Wars: Episode IV - A New Hope (1977))   
16            